<a href="https://colab.research.google.com/github/Giovasosa/Tarea-IA/blob/main/sistema_3_agentes_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Sistema Multi-Agente con RAG — Estudiantes

**Arquitectura:**
- **Agente 1 — Normalizador:** limpia, imputa, escala, codifica
- **Agente 2 — Entrenador:** embeddings + validación + selección de modelo
- **Agente 3 — Comunicador RAG:** enriquece reporte con contexto externo y responde preguntas

**Specs:** Transformers + Embeddings + RAG (FAISS) | Dataset estudiantes ~100 celdas

## 📦 Instalación

In [1]:
!pip install -q transformers sentence-transformers scikit-learn pandas numpy torch faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 42.5 MB/s eta 0:00:00


## 📊 Dataset — Estudiantes (sin normalizar)

In [2]:
import pandas as pd
import numpy as np
import io

csv_data = """nombre,edad,nota_matematica,nota_lengua,nota_ciencias,asistencia_pct,turno,rendimiento
Ana,15,8.5,9.0,7.5,95,manana,alto
Bruno,16,,6.0,5.5,70,tarde,medio
Carla,15,4.0,5.5,4.5,60,manana,bajo
Diego,17,7.0,8.0,,85,tarde,medio
Elena,16,9.5,9.5,9.0,98,manana,alto
Franco,15,3.5,4.0,3.0,50,tarde,bajo
Gisela,17,6.5,7.0,6.0,78,manana,medio
Hernan,16,8.0,,8.5,90,tarde,alto
Ivana,15,5.0,5.5,4.0,65,manana,bajo
Juan,17,9.0,8.5,9.5,96,tarde,alto
Karen,16,4.5,5.0,,55,manana,bajo
Lucas,15,7.5,7.0,7.0,82,tarde,medio
Maria,17,8.5,9.0,8.0,93,manana,alto
Nicolas,16,3.0,4.5,3.5,48,tarde,bajo
Olivia,15,,8.0,7.5,88,manana,medio
Pablo,17,6.0,6.5,6.0,75,tarde,medio
Quimey,16,9.0,9.5,9.5,99,manana,alto
Rodrigo,15,5.5,5.0,5.5,68,tarde,bajo
Sofia,17,8.0,8.5,7.5,91,manana,alto
Tomas,16,4.0,4.5,4.5,58,tarde,bajo
"""

df_raw = pd.read_csv(io.StringIO(csv_data))
print(f"Dataset: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas ({df_raw.size} celdas)")
print(f"Valores nulos: {df_raw.isnull().sum().sum()}")
df_raw.head(8)

Dataset: 20 filas x 8 columnas (160 celdas)
Valores nulos: 5


,nombre,edad,nota_matematica,nota_lengua,nota_ciencias,asistencia_pct,turno,rendimiento
0,Ana,15,8.5,9.0,7.5,95,manana,alto
1,Bruno,16,NaN,6.0,5.5,70,tarde,medio
2,Carla,15,4.0,5.5,4.5,60,manana,bajo
3,Diego,17,7.0,8.0,NaN,85,tarde,medio
4,Elena,16,9.5,9.5,9.0,98,manana,alto
5,Franco,15,3.5,4.0,3.0,50,tarde,bajo
6,Gisela,17,6.5,7.0,6.0,78,manana,medio
7,Hernan,16,8.0,NaN,8.5,90,tarde,alto


---
## 🤖 AGENTE 1 — Normalizador

In [3]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

class AgenteNormalizador:
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.log = []

    def limpiar(self, df):
        nulos = df.isnull().sum().sum()
        df = df.drop_duplicates()
        self.log.append(f"[Limpieza] Nulos encontrados: {nulos}")
        return df

    def imputar(self, df):
        for col in df.columns:
            if df[col].isnull().any():
                if df[col].dtype in ['float64', 'int64']:
                    val = df[col].median()
                    df[col] = df[col].fillna(val)
                    self.log.append(f"[Imputación] '{col}' -> mediana ({val:.2f})")
                else:
                    val = df[col].mode()[0]
                    df[col] = df[col].fillna(val)
                    self.log.append(f"[Imputación] '{col}' -> moda ('{val}')")
        return df

    def codificar(self, df, cols):
        for col in cols:
            le = LabelEncoder()
            df[col + '_enc'] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
            self.log.append(f"[Codificación] '{col}' -> {list(le.classes_)}")
        return df

    def escalar(self, df, cols):
        df[cols] = self.scaler.fit_transform(df[cols])
        self.log.append(f"[Escalado] StandardScaler en: {cols}")
        return df

    def procesar(self, df):
        print('=' * 55)
        print('AGENTE 1 — Normalizador')
        print('=' * 55)
        df = self.limpiar(df.copy())
        df = self.imputar(df)
        df = self.codificar(df, ['turno', 'rendimiento'])
        df = self.escalar(df, ['edad', 'nota_matematica', 'nota_lengua',
                                'nota_ciencias', 'asistencia_pct'])
        for e in self.log:
            print(' ', e)
        print(f'\nDataset limpio: {df.shape}')
        return df

agente1 = AgenteNormalizador()
df_limpio = agente1.procesar(df_raw)
df_limpio.head()

AGENTE 1 — Normalizador
  [Limpieza] Nulos encontrados: 5
  [Imputación] 'nota_matematica' -> mediana (6.75)
  [Imputación] 'nota_lengua' -> mediana (7.00)
  [Imputación] 'nota_ciencias' -> mediana (6.50)
  [Codificación] 'turno' -> ['manana', 'tarde']
  [Codificación] 'rendimiento' -> ['alto', 'bajo', 'medio']
  [Escalado] StandardScaler en: ['edad', 'nota_matematica', 'nota_lengua', 'nota_ciencias', 'asistencia_pct']

Dataset limpio: (20, 10)


,nombre,edad,nota_matematica,nota_lengua,nota_ciencias,asistencia_pct,turno,rendimiento,turno_enc,rendimiento_enc
0,Ana,-1.180603,1.009096,1.207528,0.541210,1.080954,manana,alto,0,0
1,Bruno,0.062137,0.114960,-0.497217,-0.514809,-0.437240,tarde,medio,1,2
2,Carla,-1.180603,-1.290109,-0.781341,-1.042819,-1.044517,manana,bajo,0,1
3,Diego,1.304877,0.242694,0.639279,0.013200,0.473676,tarde,medio,1,2
4,Elena,0.062137,1.520030,1.491652,1.333224,1.263137,manana,alto,0,0


---
## 🤖 AGENTE 2 — Entrenador

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

class AgenteEntrenador:
    def __init__(self, modelo_emb='all-MiniLM-L6-v2'):
        print('=' * 55)
        print('AGENTE 2 — Entrenador')
        print('=' * 55)
        print(f'  Cargando embedder: {modelo_emb}')
        self.embedder = SentenceTransformer(modelo_emb)
        self.mejor_modelo = None
        self.mejor_score = 0
        self.metricas = {}

    def generar_embeddings(self, df_orig):
        print('\n  [Embeddings] Generando representaciones semanticas...')
        textos = df_orig.apply(
            lambda r: f"estudiante de {r['edad']} anios, turno {r['turno']}, "
                      f"notas: matematica {r['nota_matematica']}, "
                      f"lengua {r['nota_lengua']}, ciencias {r['nota_ciencias']}, "
                      f"asistencia {r['asistencia_pct']}%",
            axis=1
        ).tolist()
        emb = self.embedder.encode(textos, show_progress_bar=False)
        print(f'  [Embeddings] Shape: {emb.shape}')
        return emb

    def construir_features(self, df, emb):
        cols = ['edad', 'nota_matematica', 'nota_lengua',
                'nota_ciencias', 'asistencia_pct', 'turno_enc']
        X = np.hstack([df[cols].values, emb])
        print(f'  [Features] {len(cols)} numericas + {emb.shape[1]} embedding = {X.shape[1]} total')
        return X

    def entrenar(self, X, y):
        print('\n  [Entrenamiento] Validacion cruzada cv=5:')
        candidatos = {
            'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=500, random_state=42)
        }
        for nombre, modelo in candidatos.items():
            scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
            m, s = scores.mean(), scores.std()
            self.metricas[nombre] = {'accuracy_media': round(m, 4), 'std': round(s, 4)}
            print(f'    {nombre}: {m:.4f} +/- {s:.4f}')
            if m > self.mejor_score:
                self.mejor_score = m
                self.mejor_nombre = nombre
                self.mejor_modelo = modelo
        self.mejor_modelo.fit(X, y)
        print(f'\n  Mejor modelo: {self.mejor_nombre} ({self.mejor_score:.4f})')

    def procesar(self, df, df_orig):
        emb = self.generar_embeddings(df_orig)
        X = self.construir_features(df, emb)
        y = df['rendimiento_enc'].values
        self.entrenar(X, y)
        return {
            'modelo': self.mejor_modelo,
            'nombre_modelo': self.mejor_nombre,
            'metricas': self.metricas,
            'mejor_accuracy': self.mejor_score,
            'X': X, 'y': y
        }

agente2 = AgenteEntrenador()
resultado = agente2.procesar(df_limpio, df_raw)

AGENTE 2 — Entrenador
  Cargando embedder: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


  [Embeddings] Generando representaciones semanticas...
  [Embeddings] Shape: (20, 384)
  [Features] 6 numericas + 384 embedding = 390 total

  [Entrenamiento] Validacion cruzada cv=5:
    Random Forest: 0.8000 +/- 0.1871
    Gradient Boosting: 0.7000 +/- 0.1871
    Logistic Regression: 0.9000 +/- 0.1225

  Mejor modelo: Logistic Regression (0.9000)


---
## 🤖 AGENTE 3 — Comunicador con RAG

Dos modos:
1. **Enriquecer el reporte** con documentos pedagógicos recuperados semánticamente
2. **Responder preguntas** sobre los datos usando FAISS

In [5]:
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Base de conocimiento pedagogico para el RAG
KNOWLEDGE_BASE = [
    "Un rendimiento academico alto se asocia con asistencia superior al 85% y notas promedio mayores a 7.5.",
    "Los estudiantes con bajo rendimiento generalmente presentan asistencia menor al 65% y promedios inferiores a 5.0.",
    "La nota de matematica tiene alta correlacion con el rendimiento general en ciencias exactas.",
    "El turno manana suele presentar mejores indices de rendimiento por factores de concentracion y descanso.",
    "La intervencion temprana en estudiantes con bajo rendimiento puede revertir la situacion en un cuatrimestre.",
    "Un modelo con accuracy superior al 85% es confiable para predecir rendimiento academico.",
    "Los embeddings semanticos capturan relaciones complejas entre variables academicas que los metodos clasicos no detectan.",
    "La validacion cruzada con 5 folds es el estandar para datasets pequenos de menos de 100 registros.",
    "El Gradient Boosting suele superar a Random Forest en datasets pequenos con variables mixtas.",
    "La imputacion por mediana es mas robusta que la media ante valores extremos en notas academicas.",
]


class AgenteComunicadorRAG:
    def __init__(self, knowledge_base):
        print('=' * 55)
        print('AGENTE 3 — Comunicador RAG')
        print('=' * 55)
        print('  [RAG] Cargando embedder...')
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')
        self.knowledge_base = knowledge_base
        self._construir_indice(knowledge_base)
        print('  [GEN] Cargando generador de texto...')
        self.generador = pipeline(
            'text-generation',
            model='sshleifer/tiny-gpt2',
            max_new_tokens=60
        )
        print('  Agente 3 RAG listo')

    def _construir_indice(self, docs):
        embeddings = self.embedder.encode(docs, show_progress_bar=False)
        dim = embeddings.shape[1]
        self.indice = faiss.IndexFlatL2(dim)
        self.indice.add(embeddings.astype('float32'))
        print(f'  [RAG] Indice FAISS: {len(docs)} docs, dim={dim}')

    def recuperar(self, query, top_k=3):
        q_emb = self.embedder.encode([query], show_progress_bar=False).astype('float32')
        _, indices = self.indice.search(q_emb, top_k)
        return [self.knowledge_base[i] for i in indices[0]]

    def responder_pregunta(self, pregunta, df_original):
        print(f'\nPregunta: {pregunta}')
        print('-' * 50)
        docs = self.recuperar(pregunta, top_k=2)
        dist = df_original['rendimiento'].value_counts().to_dict()
        stats = (
            f"Total estudiantes: {len(df_original)}. "
            f"Alto: {dist.get('alto',0)}, Medio: {dist.get('medio',0)}, Bajo: {dist.get('bajo',0)}. "
            f"Prom. matematica: {df_original['nota_matematica'].mean():.1f}. "
            f"Prom. asistencia: {df_original['asistencia_pct'].mean():.1f}%."
        )
        print(f'Datos: {stats}')
        print('Contexto RAG recuperado:')
        for i, doc in enumerate(docs, 1):
            print(f'  {i}. {doc}')

    def generar_reporte(self, resultado, df_original):
        mejor = resultado['nombre_modelo']
        acc = resultado['mejor_accuracy']
        metricas = resultado['metricas']
        query = f'modelo {mejor} accuracy {acc:.2f} clasificacion rendimiento estudiantes'
        docs_ctx = self.recuperar(query, top_k=3)
        print('\n[RAG] Documentos recuperados para el reporte:')
        for i, doc in enumerate(docs_ctx, 1):
            print(f'  {i}. {doc[:75]}...')
        dist = df_original['rendimiento'].value_counts().to_dict()
        print('\n' + '=' * 55)
        print('REPORTE FINAL — Agente 3 RAG')
        print('=' * 55)
        print(f'Dataset: {len(df_original)} estudiantes, {df_original.shape[1]} variables')
        print(f'Nulos imputados: {df_original.isnull().sum().sum()}')
        print(f'Distribucion: alto={dist.get("alto",0)}, medio={dist.get("medio",0)}, bajo={dist.get("bajo",0)}')
        print()
        print('Modelos (CV 5-fold + embeddings):')
        for nombre, m in metricas.items():
            print(f'  {nombre}: {m["accuracy_media"]} +/- {m["std"]}')
        print(f'\nModelo seleccionado: {mejor} | Accuracy: {acc:.4f} ({acc*100:.1f}%)')
        print()
        print('Contexto pedagogico recuperado por RAG:')
        for i, doc in enumerate(docs_ctx, 1):
            print(f'  {i}. {doc}')
        print('=' * 55)


agente3 = AgenteComunicadorRAG(KNOWLEDGE_BASE)
agente3.generar_reporte(resultado, df_raw)

AGENTE 3 — Comunicador RAG
  [RAG] Cargando embedder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  [RAG] Indice FAISS: 10 docs, dim=384
  [GEN] Cargando generador de texto...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  Agente 3 RAG listo

[RAG] Documentos recuperados para el reporte:
  1. Un modelo con accuracy superior al 85% es confiable para predecir rendimien...
  2. La validacion cruzada con 5 folds es el estandar para datasets pequenos de ...
  3. Un rendimiento academico alto se asocia con asistencia superior al 85% y no...

REPORTE FINAL — Agente 3 RAG
Dataset: 20 estudiantes, 8 variables
Nulos imputados: 5
Distribucion: alto=7, medio=6, bajo=7

Modelos (CV 5-fold + embeddings):
  Random Forest: 0.8 +/- 0.1871
  Gradient Boosting: 0.7 +/- 0.1871
  Logistic Regression: 0.9 +/- 0.1225

Modelo seleccionado: Logistic Regression | Accuracy: 0.9000 (90.0%)

Contexto pedagogico recuperado por RAG:
  1. Un modelo con accuracy superior al 85% es confiable para predecir rendimiento academico.
  2. La validacion cruzada con 5 folds es el estandar para datasets pequenos de menos de 100 registros.
  3. Un rendimiento academico alto se asocia con asistencia superior al 85% y notas promedio mayores a 7.5.

---
## 💬 Hacer preguntas al RAG

In [6]:
agente3.responder_pregunta(
    'Cuantos estudiantes tienen bajo rendimiento y que se puede hacer?',
    df_raw
)


Pregunta: Cuantos estudiantes tienen bajo rendimiento y que se puede hacer?
--------------------------------------------------
Datos: Total estudiantes: 20. Alto: 7, Medio: 6, Bajo: 7. Prom. matematica: 6.5. Prom. asistencia: 77.2%.
Contexto RAG recuperado:
  1. La intervencion temprana en estudiantes con bajo rendimiento puede revertir la situacion en un cuatrimestre.
  2. El turno manana suele presentar mejores indices de rendimiento por factores de concentracion y descanso.


In [7]:
agente3.responder_pregunta(
    'Es confiable el modelo para predecir rendimiento?',
    df_raw
)


Pregunta: Es confiable el modelo para predecir rendimiento?
--------------------------------------------------
Datos: Total estudiantes: 20. Alto: 7, Medio: 6, Bajo: 7. Prom. matematica: 6.5. Prom. asistencia: 77.2%.
Contexto RAG recuperado:
  1. Un modelo con accuracy superior al 85% es confiable para predecir rendimiento academico.
  2. El turno manana suele presentar mejores indices de rendimiento por factores de concentracion y descanso.


In [8]:
# Cambia esta pregunta por lo que quieras consultar
mi_pregunta = 'Que relacion hay entre asistencia y notas de matematica?'
agente3.responder_pregunta(mi_pregunta, df_raw)


Pregunta: Que relacion hay entre asistencia y notas de matematica?
--------------------------------------------------
Datos: Total estudiantes: 20. Alto: 7, Medio: 6, Bajo: 7. Prom. matematica: 6.5. Prom. asistencia: 77.2%.
Contexto RAG recuperado:
  1. La nota de matematica tiene alta correlacion con el rendimiento general en ciencias exactas.
  2. La intervencion temprana en estudiantes con bajo rendimiento puede revertir la situacion en un cuatrimestre.
